In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import numpy as np

def read_the_data(scan: str): 
    data = pd.read_csv(f"../../DeltabotController/python/scans/scan_{scan}/data.csv")
    cx = data["cx"]
    cy = data["cy"]

    with open(f"../../DeltabotController/python/scans/scan_{scan}/meta.json") as f:
        jsn = json.load(f)

    hough_params = jsn["viewer"]["hough_params"]

    print(hough_params)
    # print(jsn)
    display(data.head(10))
    return (data, cx, cy, hough_params)

In [2]:
from scipy import stats

def subplot_linear(scan, hough_params, x, y, c, y_low: int, y_high: int, fit: bool):
    x_nonan = x.dropna()
    y_nonan = y.dropna()
    fig, axs = plt.subplots(1, 3, constrained_layout=True)
    ax0 = axs[0].scatter(x,y, c=c)
    axs[0].set_ylim(y_low, y_high) # make more general
    axs[0].set_xlabel(x.name)
    axs[0].set_ylabel(y.name)

    ax1 = axs[1].scatter(x, y, c=c, label=c)
    axs[1].set_xlabel(x.name)
    axs[1].set_ylabel(y.name)
    
    # axs[1].legend()
    if fit: 
        slope_poly, intercept_poly = np.polyfit(x_nonan, y_nonan, deg=1)
        xseq = np.linspace(x_nonan.iloc[-1], x_nonan.iloc[0], num=100)
        axs[1].plot(xseq, intercept_poly + slope_poly * xseq, lw=2.5)
        print("np.polyfit slope and intercept : ", slope_poly, intercept_poly)

    ax2 = axs[2].scatter(x, y, c=c)
    axs[2].set_xlabel(x.name)
    axs[2].set_ylabel(y.name)
    
    if fit: 
        slope, intercept, r_value, p_value, std_err = stats.linregress(x_nonan,y_nonan)
        xseq = np.linspace(x_nonan.iloc[-1], x_nonan.iloc[0], num=100)
        axs[2].plot(xseq, intercept + slope * xseq, color="k", lw=2.5)
        print("stats.linregress slope and intercept", slope, intercept)

    fig.suptitle(scan + "\n" + str(hough_params))
    fig.colorbar(ax2)


In [3]:
def single_plot(scan, hough_params, x, y, c, y_low: int, y_high: int, fit: bool):
    x_nonan = x.dropna()
    y_nonan = y.dropna()
    fig, ax = plt.subplots()
    fig.suptitle(scan + "\n" + str(hough_params))
    ax0 = ax.scatter(x, y, c=c, cmap='viridis', vmin=c.min(), vmax=c.max())
    ax.set_xlabel(x.name)
    ax.set_ylabel(y.name)

    cbar = fig.colorbar(ax0)
    cbar.ax.set_ylabel(c.name, rotation=270)

    if y_low is not None and y_high is not None: 
        ax.set_ylim(y_low, y_high)
    
    if fit: 
        slope, intercept, r_value, p_value, std_err = stats.linregress(x_nonan,y_nonan)
        xseq = np.linspace(x_nonan.iloc[-1], x_nonan.iloc[0], num=100)
        ax.plot(xseq, intercept + slope * xseq, color="k", lw=2.5)
        print("stats.linregress slope and intercept", slope, intercept)

    return (fig, ax)

In [4]:
# scan = "0027_dscan" #or scan 30
# data, cx, cy, hough_params =  read_the_data(scan)
# fig, ax = single_plot(scan, hough_params, cx, cy, data["radius"], None, None, True)
